# BTL-1: Inference & Eval
## Load trained adapter on Colab T4, chat or benchmark

**Upload:** `adapter.zip` (from training notebook download)

In [ ]:
# 1. Install deps (~2min)
!pip install -q unsloth xformers transformers torch sentencepiece peft
!pip install -q accelerate bitsandbytes

In [ ]:
# 2. Upload trained adapter
from google.colab import files
import zipfile, io
print('Upload adapter.zip from training output')
uploaded = files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(uploaded[name])) as z:
            z.extractall('/content/adapter')
        print(f'Extracted to /content/adapter/')
        !ls /content/adapter/

In [ ]:
# 3. Load base model + trained adapter
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

BASE_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'
ADAPTER_PATH = '/content/adapter'
MAX_LEN = 2048

print('Loading base model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_LEN,
    dtype=torch.bfloat16,
    load_in_4bit=True,
    device_map='auto',
)

print('Loading adapter...')
from peft import PeftModel
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
model.eval()
print('Ready.')

In [ ]:
# 4. Interactive chat
def generate(prompt, max_new=512, temp=0.0):
    msgs = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc, max_new_tokens=max_new,
            temperature=temp if temp > 0 else None,
            do_sample=temp > 0,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)

while True:
    try:
        inp = input('>>> ')
    except (EOFError, KeyboardInterrupt):
        break
    if inp.lower() in ('quit', 'exit'):
        break
    print(generate(inp))
    print()

In [ ]:
# 5. Upload eval.jsonl and run BTL1 eval + CRR
from google.colab import files
import zipfile, io, json, sys
from pathlib import Path
from collections import defaultdict

print('Upload eval.jsonl')
uploaded = files.upload()

eval_rows = []
for name in uploaded:
    if name.endswith('.jsonl'):
        for line in io.TextIOWrapper(io.BytesIO(uploaded[name]), encoding='utf-8'):
            line = line.strip()
            if line: eval_rows.append(json.loads(line))

eval_rows = eval_rows[:100]
print(f'Loaded {len(eval_rows)} eval rows')

# eval_
depth_scores = defaultdict(list)
for r in eval_rows:
    depth = r.get('provenance', {}).get('source_depth', -1)
    msgs = r.get('messages', [])
    prompt_msgs = msgs[:2]
    target = msgs[2]['content'] if len(msgs) > 2 else ''
    prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    gen = tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    depth_scores[depth].append(1.0 if gen == target.strip() else 0.0)

print()
for d in sorted(depth_scores):
    s = depth_scores[d]
    print(f'  Depth {d}: {sum(s)/len(s):.3f} ({len(s)} rows)')

id_scores = depth_scores.get(1, []) + depth_scores.get(2, [])
tr_scores = depth_scores.get(3, []) + depth_scores.get(4, [])
id_acc = sum(id_scores) / max(len(id_scores), 1)
tr_acc = sum(tr_scores) / max(len(tr_scores), 1)
crr = tr_acc / max(id_acc, 1e-8)
print(f'\n  In-dist acc (d1-2): {id_acc:.4f}')
print(f'  Transfer acc (d3-4): {tr_acc:.4f}')
print(f'  CRR: {crr:.4f}')

In [ ]:
# 6. Download adapter (re-pack if you want to keep a copy)
from google.colab import files
import zipfile, os

with zipfile.ZipFile('/content/adapter-out.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk('/content/adapter'):
        for f in files:
            z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), '/content/adapter'))

files.download('/content/adapter-out.zip')